Para esta primera sesión, nos enfocaremos en convertirte en un experto en **Asincronismo y Tipado**, que es el lenguaje en el que "habla" una Fintech como Wompi para procesar miles de transacciones simultáneas sin bloquear el servidor.

---

## 📘 Clase 1: Python Moderno y Concurrencia para Fintech
**Objetivo:** Construir un simulador de procesamiento de pagos que maneje múltiples solicitudes concurrentes, utilizando tipado estático y validación robusta.

### 1. El Concepto: Event Loop y Corrutinas
En una pasarela de pagos, mientras esperas que el banco responda (I/O Bound), Python no debe quedarse de brazos cruzados. Debe procesar la siguiente transacción.



### 2. Práctica Guiada: El "Wompi Mini-Core"
Copia este esquema y complétalo siguiendo las instrucciones de los comentarios. Este ejercicio simula la validación y el envío de un pago a un banco externo.

```python
import asyncio
import time
from typing import List, Dict, Optional
from dataclasses import dataclass

# 1. Tipado Estático: Definimos la estructura de un Pago
@dataclass
class PaymentRequest:
    id: int
    amount: float
    currency: str = "COP"
    status: str = "PENDING"

# 2. Asincronismo: Simulamos la latencia de un banco externo
async def call_external_bank(payment: PaymentRequest) -> Dict[str, str]:
    print(f"DEBUG: Procesando pago {payment.id} de ${payment.amount}...")
    
    # Simulamos espera de red de 2 segundos sin bloquear el hilo principal
    await asyncio.sleep(2) 
    
    return {"id": payment.id, "result": "APPROVED", "auth_code": "123456"}

# 3. Procesamiento en Lote (Batch Processing)
async def process_all_payments(payments: List[PaymentRequest]):
    start_time = time.perf_counter()
    
    # Creamos una lista de tareas para ejecutar en paralelo
    tasks = [call_external_bank(p) for p in payments]
    
    # Ejecutamos todo concurrentemente
    results = await asyncio.gather(*tasks)
    
    end_time = time.perf_counter()
    print(f"\n✅ Finalizado: {len(results)} pagos procesados en {end_time - start_time:.2f} segundos.")
    return results

# Ejecución del script
if __name__ == "__main__":
    batch = [
        PaymentRequest(id=1, amount=50000.0),
        PaymentRequest(id=2, amount=120000.0),
        PaymentRequest(id=3, amount=3500.0)
    ]
    
    asyncio.run(process_all_payments(batch))
```

---

### 3. Tareas de Aprendizaje (Challenge)
Para que esta práctica sea de nivel "Senior", debes realizar estas modificaciones:

1.  **Manejo de Errores:** Modifica `call_external_bank` para que, si el monto es mayor a $100.000$, lance una excepción personalizada `InsufficentFundsError`. Captúrala en el loop principal sin detener los otros pagos.
2.  **Validación de Tipos con Pydantic:** Refactoriza la clase `PaymentRequest` usando la librería **Pydantic**. Asegúrate de que el campo `amount` sea obligatoriamente positivo.
3.  **Timeout:** Implementa `asyncio.wait_for` para que, si el banco tarda más de 3 segundos en responder, el pago se marque como "TIMEOUT" automáticamente.

### 4. Por qué esto te sirve para Wompi
En la semana de prueba, te evaluarán cómo manejas los fallos. Si un banco se cae, tu código no puede "morir"; debe registrar el error, liberar la conexión y seguir con el siguiente pago. Dominar `asyncio` y `try/except` en contextos asíncronos es lo que diferencia a un desarrollador Junior de un **Engineeering Mid/Senior**.

---
**¿Quieres que pasemos a revisar cómo integrar esto con una base de datos PostgreSQL de forma asíncrona (Fase 2)?**